In [0]:
from delta.tables import DeltaTable
from datetime import date, datetime
from pyspark.sql.functions import col, current_date 

#User pass date 
try:
	arrival_date=dbutils.widgets.get("arrival_date")
except Exception:
	arrival_date=date.today().strptime("%Y-%m-%d")

#User pass catalog
try:
	catalog=dbutils.widgets.get("catalog")
except Exception:
	catalog="travel_bookings"

#User pass schema
try:
	schema=dbutils.widgets.get("schema")
except Exception:
	schema="default"

In [0]:
#Read customer bronze layer table
src=spark.read.table(f"{catalog}.bronze.customer_inc").filter(col("business_date")==datetime.strptime(arrival_date, "%Y-%m-%d").date())
dim_full_name=f"{catalog}.{schema}.customer_dim" 

#Ensure table with surrogate key exists 
spark.sql(f"create schema if not exists {catalog}.{schema}")

spark.sql(f"""create table if not exists {dim_full_name} 
(
customer_sk bigint generated always as identity, customer_id int, customer_name string, customer_address string,
email string, valid_from date, valid_to date, is_current boolean
) using delta 
""")

#Check table exists or not 
try:
	if spark.catalog.tableExists(dim_full_name):
		df_target=spark.read.table(dim_full_name).filter(col("is_current")==True)
		df_update=src.alias("s").join(df_target.alias("t"), on="customer_id")\
		.filter((col("t.customer_name")!=col("s.customer_name")) | (col("t.customer_address")!=col("s.customer_address")) | (col("t.email")!=col("s.email")))\
		.select("s.customer_id", "s.customer_name", "s.customer_address", "s.email", "s.valid_from", "s.valid_to", "s.is_current")
		
		target_customer=DeltaTable.forName(spark, dim_full_name)

		target_customer.alias("t").merge(src.alias("s"), (col("t.customer_id")==col("s.customer_id")) & (col("t.is_current")==True))\
		.whenMatchedUpdate(condition=(col("t.customer_name")<>col("s.customer_name")) | (col("t.customer_address")<>col("s.customer_address")) |
		(col("t.email") <> col("s.email")), set={"valid_to": col("s.valid_from"), "is_current": lit(False)})\
		.whenNotMatchedInsert(values={"customer_id": col("s.customer_id"), "customer_name": col("s.customer_name"),
		"customer_address": col("s.customer_address"), "email": col("s.email"), "valid_from": col("s.valid_from"), "valid_to": col("s.valid_to"),"is_current": lit(True)})\
		.execute()
 
		df_update.write.format("delta").mode("append").saveAsTable(dim_full_name)
		print("SCD2 (dimensional) merge complete")

	else:
		src.write.format("delta").mode("overwrite").saveAsTable(dim_full_name)
	
except Exception as e:
	print(f"Failed to apply scd2 (dimensional) merge: {str(e)}")